In [ ]:
import os.path
import cv2
import torch
import albumentations as A
import torch.nn as nn

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(
        f"GPU count: {torch.cuda.device_count()}"
        f", CUDA version: {torch.version.cuda}"
        f", cuDNN version: {torch.backends.cudnn.version()}"
    )
elif torch.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Device: ', device)

In [ ]:
class SeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dilation=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, stride=stride, dilation=dilation, padding=padding, kernel_size=(3, 3), groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


In [ ]:
class SimpleConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0 ):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )


    def forward(self, x):
        return self.block(x)

In [ ]:
class XceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            SeparableConv(in_channels, out_channels, padding=1),
            SeparableConv(out_channels, out_channels, padding=1),
            SeparableConv(out_channels, out_channels, stride, padding)
        )
        self.skip = None
        if stride!=1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )


    def forward(self, x):
        residual = x
        if self.skip is not None:
            residual = self.skip(x)
        out = self.block(x)
        return out + residual

In [ ]:
class XceptionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([
            XceptionBlock(in_channels=64, out_channels=128, stride=2, padding=1),
            XceptionBlock(in_channels=128, out_channels=256, stride=2, padding=1),
            XceptionBlock(in_channels=256, out_channels=728, stride=2, padding=1)
        ])
        self.conv_2d_32 = SimpleConv(in_channels=3, out_channels=32, stride=2, padding=1, kernel_size=(3,3))
        self.conv_2d_64 = SimpleConv(in_channels=32, out_channels=64, kernel_size=(3,3), padding=1)
        self.middle_layers = nn.ModuleList()
        for i in range(16):
            self.middle_layers.append(XceptionBlock(in_channels=728, out_channels=728, padding=1))
        self.xception_block_1024 = XceptionBlock(in_channels=728, out_channels=1024, padding=1)
        self.sep_conv_1536_1 = SeparableConv(1024, 1536, stride=1, dilation=2, padding=2)
        self.sep_conv_1536_2 = SeparableConv(1536, 1536, stride=1, dilation=2, padding=2)
        self.sep_conv_2048_2 = SeparableConv(1536, 2048, stride=1, dilation=2, padding=2)

    def forward(self, x):
        skip_dec = None
        out = self.conv_2d_32(x)
        out = self.conv_2d_64(out)

        for i, block in enumerate(self.blocks):
            out = block(out)
            if i == 0:
                skip_dec = out
        for layer in self.middle_layers:
            out = layer(out)
        out = self.xception_block_1024(out)
        out = self.sep_conv_1536_1(out)
        out = self.sep_conv_1536_2(out)
        out = self.sep_conv_2048_2(out)

        return out, skip_dec

In [ ]:
class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, dilations, dropout_aspp):
        super().__init__()
        self.point_conv2d_2048 = SimpleConv(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        self.atrous_convs = nn.ModuleList([
            SeparableConv(in_channels=in_channels, out_channels=out_channels, dilation=rate, padding=rate)
            for rate in dilations
        ])

        self.global_avg_pooling = nn.AdaptiveAvgPool2d(1)
        self.point_conv2d_2048_2 = SimpleConv(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        self.point_conv2d_1280_3 = SimpleConv(in_channels=1280, out_channels=out_channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout_aspp)

    def forward(self, x):
        out1 = self.point_conv2d_2048(x)
        out2 = [conv(x) for conv in self.atrous_convs]

        out3 = self.global_avg_pooling(x)
        out3 = self.point_conv2d_2048_2(out3)
        out3 = nn.functional.interpolate(out3, size=x.shape[2:], mode='bilinear', align_corners=True)
        out = torch.cat([out1] + out2 + [out3], 1)
        out = self.point_conv2d_1280_3(out)
        out = self.dropout(out)# maybe remove Dropout


        return out


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, num_classes, low_level_channels=128):#in_channels = 256
        super().__init__()
        self.point_conv2d_48 = SimpleConv(in_channels=low_level_channels, out_channels=48, kernel_size=1)
        self.simple_conv2d_256_1 = SimpleConv(in_channels=304, out_channels=in_channels, kernel_size=3, padding=1)
        self.simple_conv2d_256_2 = SimpleConv(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.1)
        self.point_conv2d_final = nn.Conv2d(in_channels=in_channels, out_channels=num_classes, kernel_size=1)

    def forward(self, low_features, high_features, input_shape):
        low_out = self.point_conv2d_48(low_features)
        high_out = nn.functional.interpolate(high_features, size=low_features.shape[2:], mode='bilinear', align_corners=True)
        out = torch.cat([low_out, high_out], 1)
        out = self.simple_conv2d_256_1(out)
        out = self.simple_conv2d_256_2(out)
        out = self.dropout(out)
        out = self.point_conv2d_final(out)
        out = nn.functional.interpolate(out, size=input_shape, mode='bilinear', align_corners=True)


        return out



In [ ]:
class DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = XceptionModel()
        self.aspp = ASPP(in_channels=2048, out_channels=256, dilations=[6, 12, 18], dropout_aspp=0.5)
        self.decoder = DecoderBlock(256, num_classes=num_classes)

    def forward(self, x):
        out, low_features = self.encoder(x)
        high_features = self.aspp(out)
        out = self.decoder(low_features, high_features, x.shape[2:])

        return out